In [ ]:
%pip install pandas openpyxl ipykernel

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

PASTA_DADOS = Path.cwd() / "data" / "brutos" / "ssp"

if not PASTA_DADOS.exists():
    PASTA_DADOS = Path.cwd().parent / "data" / "brutos" / "ssp"

print("Pasta dos dados:", PASTA_DADOS)
print("Arquivos encontrados:")
for arquivo in PASTA_DADOS.glob("*.xlsx"):
    print("-", arquivo.name)
    

In [ ]:
arquivos = [
    "SPDadosCriminais_2026.xlsx",
    "CelularesSubtraidos_2026.xlsx",
    "ObjetosSubtraidos_2026.xlsx",
]

for nome in arquivos:
    caminho = PASTA_DADOS / nome

    if not caminho.exists():
        print(f"\nArquivo não encontrado: {nome}")
        continue

    excel = pd.ExcelFile(caminho)

    print(f"\nArquivo: {nome}")
    print("Abas:", excel.sheet_names)
    excel.close()

In [ ]:
nome = "ObjetosSubtraidos_2026.xlsx"
caminho = PASTA_DADOS / nome

with pd.ExcelFile(caminho) as excel:
    print("Abas disponíveis:", excel.sheet_names)

    for aba in excel.sheet_names:
        print(f"\nAba: {aba}")
        amostra = pd.read_excel(excel, sheet_name=aba, nrows=5)
        display(amostra)
        

In [ ]:
resumo_colunas = []

for nome in arquivos:
    caminho = PASTA_DADOS / nome

    if not caminho.exists():
        continue

    with pd.ExcelFile(caminho) as excel:
        for aba in excel.sheet_names:
            amostra = pd.read_excel(
                excel,
                sheet_name=aba,
                nrows=5
            )

            for coluna in amostra.columns:
                resumo_colunas.append({
                    "arquivo": nome,
                    "aba": aba,
                    "coluna": str(coluna)
                })

df_colunas = pd.DataFrame(resumo_colunas)

display(df_colunas)

In [ ]:
palavras_chave = [
    "LAT",
    "LONG",
    "DATA",
    "HORA",
    "NATUREZA",
    "RUBRICA",
    "BAIRRO",
    "MUNICIPIO",
    "BO",
    "LOGRADOURO"
]

filtro = df_colunas["coluna"].str.upper().apply(
    lambda nome: any(palavra in nome for palavra in palavras_chave)
)

colunas_importantes = df_colunas.loc[filtro]

display(colunas_importantes)

In [ ]:
comparacao = (
    colunas_importantes
    .groupby(["arquivo", "aba"])["coluna"]
    .apply(lambda colunas: ", ".join(colunas))
    .reset_index()
)

display(comparacao)


In [ ]:
from openpyxl import load_workbook
import pandas as pd

abas_dados = {
    "SPDadosCriminais_2026.xlsx": [
        "JAN-JUN_2026",
        "JUL-DEZ_2026"
    ],
    "CelularesSubtraidos_2026.xlsx": [
        "CELULAR_2026"
    ],
    "ObjetosSubtraidos_2026.xlsx": [
        "OBJETOS_01_TRIM_2026",
        "OBJETOS_02_TRIM_2026",
        "OBJETOS_03_TRIM_2026"
    ]
}

resumo = []

for nome_arquivo, abas in abas_dados.items():
    caminho = PASTA_DADOS / nome_arquivo

    if not caminho.exists():
        print(f"Arquivo não encontrado: {nome_arquivo}")
        continue

    wb = load_workbook(caminho, read_only=True, data_only=True)

    try:
        for nome_aba in abas:
            if nome_aba not in wb.sheetnames:
                print(f"Aba não encontrada: {nome_aba}")
                continue

            ws = wb[nome_aba]

            resumo.append({
                "arquivo": nome_arquivo,
                "aba": nome_aba,
                "linhas_estimadas": ws.max_row - 1,
                "colunas": ws.max_column
            })
    finally:
        wb.close()

df_resumo = pd.DataFrame(resumo)

display(df_resumo)

In [ ]:
campos_essenciais = [
    "NUM_BO",
    "ANO_BO",
    "DATA_OCORRENCIA_BO",
    "RUBRICA",
    "BAIRRO",
    "LATITUDE",
    "LONGITUDE"
]

verificacao = []

for nome_arquivo, abas in abas_dados.items():
    caminho = PASTA_DADOS / nome_arquivo

    if not caminho.exists():
        continue

    with pd.ExcelFile(caminho) as excel:
        for nome_aba in abas:
            colunas = pd.read_excel(
                excel,
                sheet_name=nome_aba,
                nrows=0
            ).columns

            verificacao.append({
                "arquivo": nome_arquivo,
                "aba": nome_aba,
                **{
                    campo: campo in colunas
                    for campo in campos_essenciais
                }
            })

df_verificacao = pd.DataFrame(verificacao)

display(df_verificacao)

In [ ]:
for nome_arquivo in [
    "CelularesSubtraidos_2026.xlsx",
    "ObjetosSubtraidos_2026.xlsx"
]:
    caminho = PASTA_DADOS / nome_arquivo

    print(f"\n{'=' * 60}")
    print(nome_arquivo)

    metodologia = pd.read_excel(
        caminho,
        sheet_name="METODOLOGIA",
        header=None
    )

    for linha in metodologia.itertuples(index=False, name=None):
        for valor in linha:
            if pd.notna(valor):
                print(str(valor))

In [ ]:
nome_arquivo = "CelularesSubtraidos_2026.xlsx"
nome_aba = "CELULAR_2026"

colunas = [
    "NOME_DELEGACIA",
    "ANO_BO",
    "NUM_BO"
]

amostra = pd.read_excel(
    PASTA_DADOS / nome_arquivo,
    sheet_name=nome_aba,
    usecols=colunas,
    nrows=10000
)

total_linhas = len(amostra)

boletins_unicos = (
    amostra
    .dropna(subset=colunas)
    .drop_duplicates(subset=colunas)
)

print("Linhas analisadas:", total_linhas)
print("Boletins distintos na amostra:", len(boletins_unicos))
print("Linhas com identificação incompleta:", amostra[colunas].isna().any(axis=1).sum())
print("Linhas adicionais com identificação repetida:", len(amostra.dropna(subset=colunas)) - len(boletins_unicos))

In [ ]:
frequencia = (
    amostra
    .dropna(subset=colunas)
    .groupby(colunas)
    .size()
    .value_counts()
    .sort_index()
    .reset_index()
)

frequencia.columns = [
    "linhas_por_boletim",
    "quantidade_de_boletins"
]

display(frequencia)

In [ ]:
colunas_tri2 = pd.read_excel(
    PASTA_DADOS / "ObjetosSubtraidos_2026.xlsx",
    sheet_name="OBJETOS_02_TRIM_2026",
    nrows=0
).columns.tolist()

colunas_tri3 = pd.read_excel(
    PASTA_DADOS / "ObjetosSubtraidos_2026.xlsx",
    sheet_name="OBJETOS_03_TRIM_2026",
    nrows=0
).columns.tolist()

print("Exclusivas do terceiro trimestre:")
print(set(colunas_tri3) - set(colunas_tri2))

print("\nExclusivas do segundo trimestre:")
print(set(colunas_tri2) - set(colunas_tri3))

In [ ]:
resumo_amostras = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["DATA_OCORRENCIA_BO", "LATITUDE", "LONGITUDE"],
            nrows=10000
        )

        datas = pd.to_datetime(
            amostra["DATA_OCORRENCIA_BO"],
            errors="coerce",
            dayfirst=True
        )

        lat = pd.to_numeric(amostra["LATITUDE"], errors="coerce")
        lon = pd.to_numeric(amostra["LONGITUDE"], errors="coerce")

        coordenadas_validas = (
            lat.between(-90, 90)
            & lon.between(-180, 180)
            & (lat != 0)
            & (lon != 0)
        )

        resumo_amostras.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "linhas_amostra": len(amostra),
            "datas_ausentes_ou_invalidas": int(datas.isna().sum()),
            "data_minima": datas.min(),
            "data_maxima": datas.max(),
            "coordenadas_basicas_validas": int(coordenadas_validas.sum()),
            "coordenadas_ausentes_ou_invalidas": int((~coordenadas_validas).sum())
        })

df_resumo_amostras = pd.DataFrame(resumo_amostras)

display(df_resumo_amostras)

In [ ]:
amostra_crimes = pd.read_excel(
    PASTA_DADOS / "SPDadosCriminais_2026.xlsx",
    sheet_name="JAN-JUN_2026",
    usecols=["RUBRICA", "NATUREZA_APURADA"],
    nrows=10000
)

print("Rubricas mais frequentes na amostra:")
display(amostra_crimes["RUBRICA"].value_counts(dropna=False).head(20))

print("\nNaturezas apuradas mais frequentes:")
display(amostra_crimes["NATUREZA_APURADA"].value_counts(dropna=False).head(20))

In [ ]:
resumo_anos = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["DATA_OCORRENCIA_BO"],
            nrows=10000
        )

        datas = pd.to_datetime(
            amostra["DATA_OCORRENCIA_BO"],
            errors="coerce",
            dayfirst=True
        )

        resumo_anos.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "antes_de_2026": int((datas.dt.year < 2026).sum()),
            "em_2026": int((datas.dt.year == 2026).sum()),
            "apos_2026": int((datas.dt.year > 2026).sum()),
            "datas_invalidas": int(datas.isna().sum())
        })

display(pd.DataFrame(resumo_anos))

In [ ]:
resumo_localizacao = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["LATITUDE", "LONGITUDE"],
            nrows=10000
        )

        lat = pd.to_numeric(amostra["LATITUDE"], errors="coerce")
        lon = pd.to_numeric(amostra["LONGITUDE"], errors="coerce")

        plausivel_sp = (
            lat.between(-24.1, -23.3)
            & lon.between(-47.0, -46.3)
        )

        resumo_localizacao.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "amostra": len(amostra),
            "coordenadas_na_area_aproximada": int(plausivel_sp.sum()),
            "fora_da_area_ou_invalidas": int((~plausivel_sp).sum())
        })

display(pd.DataFrame(resumo_localizacao))

In [ ]:
resumo_localidades = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["NOME_MUNICIPIO", "BAIRRO"],
            nrows=10000
        )

        municipio = (
            amostra["NOME_MUNICIPIO"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        bairro = (
            amostra["BAIRRO"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        resumo_localidades.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "linhas_amostra": len(amostra),
            "municipio_sao_paulo": int((municipio == "SÃO PAULO").sum()),
            "bairro_vila_mariana": int((bairro == "VILA MARIANA").sum()),
            "municipio_ausente": int(municipio.isna().sum()),
            "bairro_ausente": int(bairro.isna().sum())
        })

display(pd.DataFrame(resumo_localidades))

In [ ]:
amostra_municipios = pd.read_excel(
    PASTA_DADOS / "SPDadosCriminais_2026.xlsx",
    sheet_name="JAN-JUN_2026",
    usecols=["NOME_MUNICIPIO"],
    nrows=10000
)

display(
    amostra_municipios["NOME_MUNICIPIO"]
    .value_counts(dropna=False)
    .head(15)
)

In [ ]:
import sys
import importlib.util

print("Python do notebook:", sys.executable)
print("GeoPandas disponível:", importlib.util.find_spec("geopandas") is not None)

In [ ]:
%pip install geopandas

In [ ]:
import geopandas as gpd

print("Versão do GeoPandas:", gpd.__version__) 

In [ ]:
from pathlib import Path
import geopandas as gpd

raiz = Path.cwd()

if not (raiz / "data" / "brutos").exists():
    raiz = raiz.parent

caminho_iluminacao = (
    raiz / "data" / "brutos" / "geosampa"
    / "SIRGAS_GPKG_iluminacaopublica.gpkg"
)

if not caminho_iluminacao.is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_iluminacao}")

iluminacao = gpd.read_file(
    caminho_iluminacao,
    layer="SIRGAS_GPKG_iluminacaopublica"
)

print("Linhas e colunas:", iluminacao.shape)
print("Colunas:", iluminacao.columns.tolist())
print("Sistema de coordenadas:", iluminacao.crs)

print("\nTipos de geometria:")
print(iluminacao.geom_type.value_counts(dropna=False))

print("\nTipos das colunas:")
print(iluminacao.dtypes)

print("\nNulos por coluna:")
print(iluminacao.isna().sum())

display(iluminacao.head())

In [ ]:
colunas_texto = [
    "il_placa",
    "il_tipilum",
    "il_enderec",
    "il_unidade"
]

print("Textos vazios ou contendo apenas espaços:")
for coluna in colunas_texto:
    vazios = iluminacao[coluna].astype("string").str.strip().eq("")
    print(f"{coluna}: {int(vazios.sum())}")

print("\nGeometrias vazias:", int(iluminacao.geometry.is_empty.sum()))
print("Geometrias inválidas:", int((~iluminacao.geometry.is_valid).sum()))

print("\nTipos de iluminação:")
print(iluminacao["il_tipilum"].value_counts(dropna=False))

In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd()

if not (raiz / "src").is_dir():
    raiz = raiz.parent

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.dados import ler_ssp, ler_iluminacao

amostra_ssp = ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026",
    colunas=["DATA_OCORRENCIA_BO", "LATITUDE", "LONGITUDE"],
    n_linhas=5
)

amostra_luz = ler_iluminacao(
    raiz / "data/brutos/geosampa/SIRGAS_GPKG_iluminacaopublica.gpkg",
    n_linhas=5
)

print("SSP — linhas e colunas:", amostra_ssp.shape)
print("Iluminação — linhas e colunas:", amostra_luz.shape)
print("Iluminação — CRS:", amostra_luz.crs)
print("Iluminação — geometrias:", amostra_luz.geom_type.tolist())

In [ ]:
import pandas as pd

fontes_dicionario = {
    "SPDadosCriminais_2026.xlsx": "Campos da Tabela_SPDADOS",
    "CelularesSubtraidos_2026.xlsx": "DICIONARIO DE DADOS",
    "ObjetosSubtraidos_2026.xlsx": "DICIONARIO DE DADOS"
}

campos_interesse = {
    "NOME_DELEGACIA", "ANO_BO", "NUM_BO", "VERSAO",
    "DATA_OCORRENCIA_BO", "DATA_REGISTRO",
    "DATAHORA_REGISTRO_BO", "RUBRICA", "NATUREZA_APURADA",
    "LATITUDE", "LONGITUDE", "LOGRADOURO", "BAIRRO",
    "NOME_MUNICIPIO", "NOME_MUNICIPIO_CIRCUNSCRICAO",
    "NOME_MUNICIPIO_CIRC", "CIDADE",
    "DESCR_TIPOLOCAL", "DESCR_SUBTIPOLOCAL"
}

for nome_arquivo, aba in fontes_dicionario.items():
    tabela = pd.read_excel(
        raiz / "data" / "brutos" / "ssp" / nome_arquivo,
        sheet_name=aba,
        header=None,
        usecols=[0, 1]
    )

    tabela.columns = ["campo", "descricao"]
    tabela["campo"] = (
        tabela["campo"].astype("string").str.strip().str.upper()
    )

    print(f"\nArquivo: {nome_arquivo}")
    print(
        tabela.loc[tabela["campo"].isin(campos_interesse)]
        .to_string(index=False)
    )
    

In [ ]:
campos_geral = [
    "NOME_DELEGACIA",
    "ANO_BO",
    "NUM_BO",
    "DATA_REGISTRO",
    "DATA_OCORRENCIA_BO",
    "RUBRICA",
    "NATUREZA_APURADA",
    "DESCR_TIPOLOCAL",
    "DESCR_SUBTIPOLOCAL",
    "BAIRRO",
    "LOGRADOURO",
    "LATITUDE",
    "LONGITUDE",
    "NOME_MUNICIPIO",
    "NOME_MUNICIPIO_CIRCUNSCRICAO"
]

amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026",
    colunas=campos_geral,
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Arquivo: SPDadosCriminais_2026.xlsx")
print("Aba: JAN-JUN_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())

In [ ]:
amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JUL-DEZ_2026",
    colunas=campos_geral,
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Arquivo: SPDadosCriminais_2026.xlsx")
print("Aba: JUL-DEZ_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())

In [ ]:
campos_celulares_objetos = [
    "NOME_DELEGACIA",
    "ANO_BO",
    "NUM_BO",
    "VERSAO",
    "DATAHORA_REGISTRO_BO",
    "DATA_OCORRENCIA_BO",
    "RUBRICA",
    "DESCR_TIPOLOCAL",
    "DESCR_SUBTIPOLOCAL",
    "BAIRRO",
    "LOGRADOURO",
    "LATITUDE",
    "LONGITUDE",
    "NOME_MUNICIPIO",
    "NOME_MUNICIPIO_CIRC"
]

amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/CelularesSubtraidos_2026.xlsx",
    aba="CELULAR_2026",
    colunas=campos_celulares_objetos + ["CIDADE"],
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Arquivo: CelularesSubtraidos_2026.xlsx")
print("Aba: CELULAR_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())

In [ ]:
abas_objetos = [
    "OBJETOS_01_TRIM_2026",
    "OBJETOS_02_TRIM_2026",
    "OBJETOS_03_TRIM_2026"
]

for aba in abas_objetos:
    amostra_a2 = ler_ssp(
        raiz / "data/brutos/ssp/ObjetosSubtraidos_2026.xlsx",
        aba=aba,
        colunas=campos_celulares_objetos,
        n_linhas=10000
    )

    resumo_a2 = pd.DataFrame({
        "tipo_observado": amostra_a2.dtypes.astype(str),
        "nulos": amostra_a2.isna().sum(),
        "textos_em_branco": [
            int(
                amostra_a2[coluna]
                .astype("string")
                .str.strip()
                .eq("")
                .sum()
            )
            for coluna in amostra_a2.columns
        ]
    })

    print(f"\nAba: {aba}")
    print("Linhas da amostra:", len(amostra_a2))
    print(resumo_a2.to_string())

    del amostra_a2

In [ ]:
print("Iniciando leitura do terceiro trimestre...", flush=True)

amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/ObjetosSubtraidos_2026.xlsx",
    aba="OBJETOS_03_TRIM_2026",
    colunas=campos_celulares_objetos,
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Aba: OBJETOS_03_TRIM_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())
print("Inspeção da amostra finalizada.")